In [ ]:
# notebooks/03_Test.ipynb

# 1. Environment Initialization
import os
import shutil
import sys
import random
import itertools
import numpy as np
import pandas as pd
import torch
from torch.amp import autocast
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q monai
from monai.inferers import sliding_window_inference

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.3 MB/s eta 0:00:00


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

PROJECT_ROOT = "/content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor"
sys.path.append(PROJECT_ROOT)

import src.config as config
from src.dataset import get_brats_dataloaders
from src.metrics import SegmentationMetrics
from src.models.mamba_backbone import MambaBackbone, SharedDeepMambaBackbone
from src.models.fusion import PresenceAwareCrossModalFusion
from src.models.decoder import SegmentationDecoder3D, AuxiliaryDecoder3D

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
DATASET_ZIP = "/content/drive/MyDrive/ML-Datasets/BraTS2020_TrainingData_ABC.zip"
LOCAL_EXTRACT_DIR = "/content/MICCAI_BraTS2020_TrainingData_ABC"
LOCAL_DATA_DIR = LOCAL_EXTRACT_DIR

# Verify archive existence immediately before runtime allocation
assert os.path.exists(DATASET_ZIP), f"Dataset archive not found: {DATASET_ZIP}"

# Robust extraction guard: triggers if directory does not exist or is completely empty
if not os.path.exists(LOCAL_DATA_DIR) or len(os.listdir(LOCAL_DATA_DIR)) == 0:
    print(f"Extracting preprocessed dataset to fast local runtime storage: {LOCAL_EXTRACT_DIR}...")
    os.makedirs(LOCAL_EXTRACT_DIR, exist_ok=True)
    shutil.unpack_archive(DATASET_ZIP, LOCAL_EXTRACT_DIR, "zip")
    print("Extraction complete. Preprocessed dataset ready for I/O operations.")
else:
    print("Valid local preprocessed dataset cache detected. Skipping extraction.")

Extracting preprocessed dataset to fast local runtime storage: /content/MICCAI_BraTS2020_TrainingData_ABC...
Extraction complete. Preprocessed dataset ready for I/O operations.


In [5]:
# 2. Pipeline Dataset Loaders Construction
_, val_loader = get_brats_dataloaders()

# 3. Structural Module Instantiations & Checkpoint Loading
backbone = MambaBackbone(embed_dim=config.EMBED_DIM).to(device)
fusion = PresenceAwareCrossModalFusion(embed_dim=config.EMBED_DIM).to(device)
shared_backbone = SharedDeepMambaBackbone(embed_dim=config.EMBED_DIM).to(device)
decoder = SegmentationDecoder3D(embed_dim=config.EMBED_DIM, out_channels=config.NUM_SEG_CLASSES).to(device)
aux_decoder = AuxiliaryDecoder3D(embed_dim=config.EMBED_DIM, out_channels=config.NUM_SEG_CLASSES).to(device)

best_seg_path = os.path.join(config.CHECKPOINT_DIR, "best_seg.pth")
assert os.path.exists(best_seg_path), f"Checkpoint not found: {best_seg_path}"

checkpoint = torch.load(best_seg_path, map_location=device)
backbone.load_state_dict(checkpoint["backbone_state"])
fusion.load_state_dict(checkpoint["fusion_state"])
shared_backbone.load_state_dict(checkpoint["shared_backbone_state"])
decoder.load_state_dict(checkpoint["decoder_state"])
aux_decoder.load_state_dict(checkpoint["aux_decoder_state"])

backbone.eval()
fusion.eval()
shared_backbone.eval()
decoder.eval()

# 4. Define 15 Modality Combinations
# 0: T1, 1: T1ce, 2: T2, 3: FLAIR
mod_idx = [0, 1, 2, 3]
mod_names = ["T1", "T1ce", "T2", "FLAIR"]

combinations = []
for r in range(1, 5):
    combinations.extend(list(itertools.combinations(mod_idx, r)))

results = []

# 5. Evaluation Loop across 15 settings
print(f"Starting robust multi-modality evaluation across {len(combinations)} settings...")

with torch.no_grad():
    for comb in combinations:
        comb_names = "+".join([mod_names[i] for i in comb])
        print(f"\n--- Testing Modalities: {comb_names} ---")

        keep_mask = torch.zeros(4, device=device)
        keep_mask[list(comb)] = 1.0

        seg_tracker = SegmentationMetrics()

        for batch in tqdm(val_loader, desc=f"Eval {comb_names}", leave=False):
            images = batch["image"].to(device)
            seg_targets = batch["label"].to(device)
            B_current = images.size(0)
            batch_seg_logits = []

            for b in range(B_current):
                single_img = images[b:b+1]

                # def evaluation_predictor(patch_images):
                #     modality_tokens, spatial_shape, skip_features, _ = backbone(patch_images)

                #     processed_modality_tokens = []
                #     processed_images = patch_images.clone()

                #     for i in range(4):
                #         if keep_mask[i] == 1.0:
                #             processed_modality_tokens.append(modality_tokens[i])
                #         else:
                #             processed_modality_tokens.append(torch.zeros_like(modality_tokens[i]))
                #             processed_images[:, i, :, :, :] = 0.0

                #     fused_tokens = fusion(processed_modality_tokens, processed_images)
                #     latent_tokens = shared_backbone(fused_tokens)
                #     seg_logits = decoder(latent_tokens, spatial_shape, skip_features)

                #     return seg_logits

                def evaluation_predictor(patch_images):
                    # 1. Apply dropout to the raw input images FIRST
                    processed_images = patch_images.clone()
                    for i in range(4):
                        if keep_mask[i] == 0.0:
                            processed_images[:, i, :, :, :] = 0.0

                    # 2. Backbone now processes genuinely incomplete inputs
                    modality_tokens, spatial_shape, skip_features, _ = backbone(processed_images)

                    # 3. Format tokens for the fusion module as required
                    processed_modality_tokens = []
                    for i in range(4):
                        if keep_mask[i] == 1.0:
                            processed_modality_tokens.append(modality_tokens[i])
                        else:
                            processed_modality_tokens.append(torch.zeros_like(modality_tokens[i]))

                    fused_tokens = fusion(processed_modality_tokens, processed_images)
                    latent_tokens = shared_backbone(fused_tokens)
                    seg_logits = decoder(latent_tokens, spatial_shape, skip_features)

                    return seg_logits

                with autocast(device_type=device.type, enabled=(device.type == "cuda")):
                    seg_logits = sliding_window_inference(
                        inputs=single_img,
                        roi_size=config.PATCH_SIZE,
                        sw_batch_size=1,
                        predictor=evaluation_predictor,
                        overlap=0.25,
                        mode="gaussian"
                    )
                batch_seg_logits.append(seg_logits)

            seg_logits = torch.cat(batch_seg_logits, dim=0)
            seg_preds = torch.argmax(seg_logits, dim=1, keepdim=True)
            seg_tracker.update(seg_preds, seg_targets, run_hd=False)

        metrics = seg_tracker.compute(run_hd=False)
        mean_dice = (metrics["dice_WT"] + metrics["dice_TC"] + metrics["dice_ET"]) / 3.0

        results.append({
            "Missing Modalities": "+".join([mod_names[i] for i in mod_idx if i not in comb]) or "None",
            "Present Modalities": comb_names,
            "Dice WT": round(metrics["dice_WT"], 4),
            "Dice TC": round(metrics["dice_TC"], 4),
            "Dice ET": round(metrics["dice_ET"], 4),
            "Mean Dice": round(mean_dice, 4),
            "HD95 WT": round(metrics["hd95_WT"], 4) if "hd95_WT" in metrics else "N/A",
            "HD95 TC": round(metrics["hd95_TC"], 4) if "hd95_TC" in metrics else "N/A",
            "HD95 ET": round(metrics["hd95_ET"], 4) if "hd95_ET" in metrics else "N/A"
        })
        print(f"Mean Dice: {mean_dice:.4f} | WT: {metrics['dice_WT']:.4f}, TC: {metrics['dice_TC']:.4f}, ET: {metrics['dice_ET']:.4f}")

# 6. Formatting and Saving Results
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by="Mean Dice", ascending=False).reset_index(drop=True)

display(df_results)
csv_out_path = os.path.join(config.CHECKPOINT_DIR, "modality_dropout_results.csv")
df_results.to_csv(csv_out_path, index=False)

Starting robust multi-modality evaluation across 15 settings...

--- Testing Modalities: T1 ---


Mean Dice: 0.0004 | WT: 0.0006, TC: 0.0005, ET: 0.0002

--- Testing Modalities: T1ce ---


Mean Dice: 0.4806 | WT: 0.3728, TC: 0.5465, ET: 0.5224

--- Testing Modalities: T2 ---


Mean Dice: 0.3678 | WT: 0.5736, TC: 0.4346, ET: 0.0953

--- Testing Modalities: FLAIR ---


Mean Dice: 0.3281 | WT: 0.6728, TC: 0.2967, ET: 0.0149

--- Testing Modalities: T1+T1ce ---


...

Mean Dice: 0.7840 | WT: 0.8469, TC: 0.7444, ET: 0.7608

--- Testing Modalities: T1+T2+FLAIR ---


Mean Dice: 0.3752 | WT: 0.8587, TC: 0.2666, ET: 0.0002

--- Testing Modalities: T1ce+T2+FLAIR ---


Mean Dice: 0.6747 | WT: 0.8342, TC: 0.6093, ET: 0.5806

--- Testing Modalities: T1+T1ce+T2+FLAIR ---


Mean Dice: 0.8302 | WT: 0.8936, TC: 0.8111, ET: 0.7860


,Missing Modalities,Present Modalities,Dice WT,Dice TC,Dice ET,Mean Dice,HD95 WT,HD95 TC,HD95 ET
0,None,T1+T1ce+T2+FLAIR,0.8936,0.8111,0.7860,0.8302,0.0,0.0,0.0
1,T2,T1+T1ce+FLAIR,0.8469,0.7444,0.7608,0.7840,0.0,0.0,0.0
2,T1+T2,T1ce+FLAIR,0.8084,0.6376,0.5932,0.6797,0.0,0.0,0.0
3,T1,T1ce+T2+FLAIR,0.8342,0.6093,0.5806,0.6747,0.0,0.0,0.0
4,T1+FLAIR,T1ce+T2,0.6333,0.6347,0.6390,0.6357,0.0,0.0,0.0
5,T1+T2+FLAIR,T1ce,0.3728,0.5465,0.5224,0.4806,0.0,0.0,0.0
6,FLAIR,T1+T1ce+T2,0.3619,0.5102,0.4799,0.4507,0.0,0.0,0.0
7,T1+T1ce,T2+FLAIR,0.8261,0.4587,0.0212,0.4353,0.0,0.0,0.0
8,T1ce,T1+T2+FLAIR,0.8587,0.2666,0.0002,0.3752,0.0,0.0,0.0
9,T1+T1ce+FLAIR,T2,0.5736,0.4346,0.0953,0.3678,0.0,0.0,0.0
